In [ ]:
# ==============================
# 1. Import libraries
# ==============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# ==============================
# 2. Load dataset
# ==============================

df = pd.read_csv("/content/brca_data_w_subtypes.csv")
# df_valid = pd.read_csv("/content/Breast Cancer METABRIC.csv")

print(df.shape)
df.head()

(705, 1941)


,rs_CLEC3A,rs_CPB1,rs_SCGB2A2,rs_SCGB1D2,rs_TFF1,rs_MUCL1,rs_GSTM1,rs_PIP,rs_ADIPOQ,rs_ADH1B,...,pp_p62.LCK.ligand,pp_p70S6K,pp_p70S6K.pT389,pp_p90RSK,pp_p90RSK.pT359.S363,vital.status,PR.Status,ER.Status,HER2.Final.Status,histological.type
0,0.892818,6.580103,14.123672,10.606501,13.189237,6.649466,10.520335,10.338490,10.248379,10.229970,...,-0.691766,-0.337863,-0.178503,0.011638,-0.207257,0,Positive,Positive,Negative,infiltrating ductal carcinoma
1,0.000000,3.691311,17.116090,15.517231,9.867616,9.691667,8.179522,7.911723,1.289598,1.818891,...,0.279067,0.292925,-0.155242,-0.089365,0.267530,0,Positive,Negative,Negative,infiltrating ductal carcinoma
2,3.748150,4.375255,9.658123,5.326983,12.109539,11.644307,10.517330,5.114925,11.975349,11.911437,...,0.219910,0.308110,-0.190794,-0.222150,-0.198518,0,Positive,Positive,Negative,infiltrating ductal carcinoma
3,0.000000,18.235519,18.535480,14.533584,14.078992,8.913760,10.557465,13.304434,8.205059,9.211476,...,-0.266554,-0.079871,-0.463237,0.522998,-0.046902,0,Positive,Positive,Negative,infiltrating ductal carcinoma
4,0.000000,4.583724,15.711865,12.804521,8.881669,8.430028,12.964607,6.806517,4.294341,5.385714,...,-0.441542,-0.152317,0.511386,-0.096482,0.037473,0,Positive,Positive,Negative,infiltrating ductal carcinoma


In [ ]:
# ==============================
# 3. Select features and target
# ==============================

rs_cols = [col for col in df.columns if col.startswith("rs_")]

X = df[rs_cols]
y = df["ER.Status"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (705, 604)
Target shape: (705,)


In [ ]:
# ==============================
# 4. Check target values before cleaning
# ==============================

print(y.value_counts(dropna=False))

ER.Status
Positive                       414
Negative                       135
NaN                            122
Not Performed                   27
Performed but Not Available      5
Indeterminate                    2
Name: count, dtype: int64


In [ ]:
# ==============================
# 5. Clean target labels
# Remove NaN, Indeterminate, Not Performed, etc.
# ==============================

valid_classes = ["Positive", "Negative"]

valid_rows = (
    y.notna() &
    y.isin(valid_classes)
)

X_clean = X.loc[valid_rows].copy()
y_clean = y.loc[valid_rows].copy()

print("Original rows:", len(y))
print("Cleaned rows:", len(y_clean))
print("Removed rows:", len(y) - len(y_clean))

print("\nCleaned target counts:")
print(y_clean.value_counts())

print("\nCleaned target proportions:")
print(y_clean.value_counts(normalize=True))

Original rows: 705
Cleaned rows: 549
Removed rows: 156

Cleaned target counts:
ER.Status
Positive    414
Negative    135
Name: count, dtype: int64

Cleaned target proportions:
ER.Status
Positive    0.754098
Negative    0.245902
Name: proportion, dtype: float64


In [ ]:
# ==============================
# 6. Encode labels
# ==============================

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_clean)

print("Label mapping:")
for label, code in zip(encoder.classes_, encoder.transform(encoder.classes_)):
    print(label, "=", code)

Label mapping:
Negative = 0
Positive = 1


In [ ]:
# ==============================
# 7. Train-test split
# Stratify keeps the same class ratio
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (439, 604)
X_test: (110, 604)
y_train: (439,)
y_test: (110,)


In [ ]:
# ==============================
# 8. Check class balance after split
# ==============================

print("Training class distribution:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts(normalize=True))

Training class distribution:
1    0.753986
0    0.246014
Name: proportion, dtype: float64

Testing class distribution:
1    0.754545
0    0.245455
Name: proportion, dtype: float64


In [ ]:
# ==============================
# 9. Feature selection
# IMPORTANT: fit only on training data
# ==============================

selector = SelectKBest(score_func=f_classif, k=300)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

print("Selected train shape:", X_train_selected.shape)
print("Selected test shape:", X_test_selected.shape)

Selected train shape: (439, 300)
Selected test shape: (110, 300)


In [ ]:
# ==============================
# 10. Train model
# class_weight helps with imbalance
# ==============================

model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train_selected, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score
)

# ==============================
# 1. Predictions
# ==============================

y_pred = model.predict(X_test_selected)

# predicted probabilities
y_proba = model.predict_proba(X_test_selected)[:, 1]

In [ ]:
# ==============================
# 2. Basic evaluation
# ==============================

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

Confusion Matrix:
[[20  7]
 [ 4 79]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.83      0.74      0.78        27
    Positive       0.92      0.95      0.93        83

    accuracy                           0.90       110
   macro avg       0.88      0.85      0.86       110
weighted avg       0.90      0.90      0.90       110



In [ ]:
# ==============================
# 3. Individual metrics
# ==============================

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_proba)

print("Accuracy:", accuracy)
print("Balanced Accuracy:", balanced_acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

Accuracy: 0.9
Balanced Accuracy: 0.8462739848282017
Precision: 0.9186046511627907
Recall: 0.9518072289156626
F1-score: 0.9349112426035503
ROC-AUC: 0.9136546184738956


In [ ]:

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)
rf_model.fit(X_train_selected, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=3, min_samples_split=5,
                       n_estimators=300, random_state=42)

In [ ]:
y_pred = rf_model.predict(X_test_selected)
y_proba = rf_model.predict_proba(X_test_selected)[:, 1]
print(confusion_matrix(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[20  7]
 [ 5 78]]
              precision    recall  f1-score   support

    Negative       0.80      0.74      0.77        27
    Positive       0.92      0.94      0.93        83

    accuracy                           0.89       110
   macro avg       0.86      0.84      0.85       110
weighted avg       0.89      0.89      0.89       110

Accuracy: 0.8909090909090909
Balanced Accuracy: 0.8402498884426595
Precision: 0.9176470588235294
Recall: 0.9397590361445783
F1-score: 0.9285714285714286
ROC-AUC: 0.9134315037929497


In [ ]:
import numpy as np

negative_count = np.sum(y_train == 0)
positive_count = np.sum(y_train == 1)

scale_pos_weight = negative_count / positive_count

print(scale_pos_weight)

0.32628398791540786


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train_selected, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred = xgb_model.predict(X_test_selected)
y_proba = xgb_model.predict_proba(X_test_selected)[:, 1]

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score
)

print(confusion_matrix(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[21  6]
 [ 9 74]]
              precision    recall  f1-score   support

    Negative       0.70      0.78      0.74        27
    Positive       0.93      0.89      0.91        83

    accuracy                           0.86       110
   macro avg       0.81      0.83      0.82       110
weighted avg       0.87      0.86      0.87       110

Accuracy: 0.8636363636363636
Balanced Accuracy: 0.8346720214190093
Precision: 0.925
Recall: 0.891566265060241
F1-score: 0.9079754601226994
ROC-AUC: 0.92681838464971


The selected XGBoost hyperparameters were chosen to create a balanced and stable model for a high-dimensional genomics dataset with a relatively small sample size and class imbalance.



A smaller tree depth was used to reduce overfitting and improve generalization since deeper trees can memorize noisy gene expression patterns instead of learning meaningful biological relationships.




A lower learning rate was selected to allow the model to learn gradually and more stably rather than making overly aggressive updates. The number of estimators was kept moderate to provide sufficient learning capacity while avoiding excessive model complexity.
 A higher minimum child weight was used to prevent unreliable splits formed from very small subsets of samples, which is important in genomic datasets where noise can easily influence tree construction. Subsampling was applied so that each tree learns from only a portion of the training samples, improving robustness and reducing overfitting.



 Column sampling was also used so that each tree considers only a subset of gene features, which is especially useful in high-dimensional biological data containing many potentially noisy or redundant variables.



 Additionally, scale_pos_weight was included to address class imbalance by giving greater importance to the minority class during training, helping the model avoid bias toward the majority class.

In [ ]:
from xgboost import XGBClassifier

xgb_model_changed = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.5,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

xgb_model_changed.fit(X_train_selected, y_train)
y_pred = xgb_model_changed.predict(X_test_selected)
y_proba = xgb_model_changed.predict_proba(X_test_selected)[:, 1]
print(confusion_matrix(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[21  6]
 [ 8 75]]
              precision    recall  f1-score   support

    Negative       0.72      0.78      0.75        27
    Positive       0.93      0.90      0.91        83

    accuracy                           0.87       110
   macro avg       0.83      0.84      0.83       110
weighted avg       0.88      0.87      0.87       110

Accuracy: 0.8727272727272727
Balanced Accuracy: 0.8406961178045516
Precision: 0.9259259259259259
Recall: 0.9036144578313253
F1-score: 0.9146341463414634
ROC-AUC: 0.9348505131637661


In [ ]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)

The RBF kernel allows SVM to capture nonlinear relationships between genes and ER status.

The balanced class weight helps the model pay more attention to the minority class during training without changing dataset size

A moderate C value helps prevent overfitting while still allowing the model to learn meaningful class boundaries.

Gamma set to scale automatically adapts to the feature variance and is usually a safe starting point for high-dimensional biological data

In [ ]:
svm_model.fit(X_train_selected, y_train)
y_pred = svm_model.predict(X_test_selected)

y_proba = svm_model.predict_proba(X_test_selected)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[20  7]
 [ 5 78]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.80      0.74      0.77        27
    Positive       0.92      0.94      0.93        83

    accuracy                           0.89       110
   macro avg       0.86      0.84      0.85       110
weighted avg       0.89      0.89      0.89       110

Accuracy: 0.8909090909090909
Balanced Accuracy: 0.8402498884426595
Precision: 0.9176470588235294
Recall: 0.9397590361445783
F1-score: 0.9285714285714286
ROC-AUC: 0.8821954484605087


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

In [ ]:
svm_model.fit(X_train_scaled, y_train)
y_pred = svm_model.predict(X_test_scaled)

y_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[20  7]
 [ 5 78]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.80      0.74      0.77        27
    Positive       0.92      0.94      0.93        83

    accuracy                           0.89       110
   macro avg       0.86      0.84      0.85       110
weighted avg       0.89      0.89      0.89       110

Accuracy: 0.8909090909090909
Balanced Accuracy: 0.8402498884426595
Precision: 0.9176470588235294
Recall: 0.9397590361445783
F1-score: 0.9285714285714286
ROC-AUC: 0.8844265952699688


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

In [ ]:
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42)

In [ ]:
y_pred = lr_model.predict(X_test_scaled)

y_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[19  8]
 [10 73]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.66      0.70      0.68        27
    Positive       0.90      0.88      0.89        83

    accuracy                           0.84       110
   macro avg       0.78      0.79      0.78       110
weighted avg       0.84      0.84      0.84       110

Accuracy: 0.8363636363636363
Balanced Accuracy: 0.7916108879964301
Precision: 0.9012345679012346
Recall: 0.8795180722891566
F1-score: 0.8902439024390244
ROC-AUC: 0.8616688978134761


ER subtype relationships are not purely linear

one global linear boundary


But genomics data often contains:

nonlinear gene interactions
complex feature relationships
correlated biological pathways

Tree models and SVMs can capture these better.

VERY STRONG BOOSTING MODEL

In [ ]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    num_leaves=7,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.5,
    class_weight="balanced",
    random_state=42
)

lgbm_model.fit(X_train_selected, y_train)

y_pred = lgbm_model.predict(X_test_selected)
y_proba = lgbm_model.predict_proba(X_test_selected)[:, 1]

[LightGBM] [Info] Number of positive: 331, number of negative: 108
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002187 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41085
[LightGBM] [Info] Number of data points in the train set: 439, number of used features: 300
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[21  6]
 [ 7 76]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.78      0.76        27
    Positive       0.93      0.92      0.92        83

    accuracy                           0.88       110
   macro avg       0.84      0.85      0.84       110
weighted avg       0.88      0.88      0.88       110

Accuracy: 0.8818181818181818
Balanced Accuracy: 0.8467202141900937
Precision: 0.926829268292683
Recall: 0.9156626506024096
F1-score: 0.9212121212121213
ROC-AUC: 0.9147701918786256


Scientifically this is actually a strong result
**tree ensemble methods dominate linear models**
We are discovering that:


for this ER genomics problem

which is biologically plausible because gene interactions are often nonlinear.

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.0 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.05,
    depth=3,
    l2_leaf_reg=5,
    auto_class_weights="Balanced",
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=0
)

cat_model.fit(X_train_selected, y_train)

y_pred = cat_model.predict(X_test_selected)
y_proba = cat_model.predict_proba(X_test_selected)[:, 1]

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[21  6]
 [ 7 76]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.78      0.76        27
    Positive       0.93      0.92      0.92        83

    accuracy                           0.88       110
   macro avg       0.84      0.85      0.84       110
weighted avg       0.88      0.88      0.88       110

Accuracy: 0.8818181818181818
Balanced Accuracy: 0.8467202141900937
Precision: 0.926829268292683
Recall: 0.9156626506024096
F1-score: 0.9212121212121213
ROC-AUC: 0.9344042838018741


| Model               | Main Strength                      |
| ------------------- | ---------------------------------- |
| Random Forest       | strongest practical classification |
| CatBoost            | strongest overall balance          |
| XGBoost             | strongest probability separation   |
| LightGBM            | very balanced                      |
| SVM                 | strong boundary classification     |
| Logistic Regression | weakest overall                    |


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [ ]:
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def run_cv_for_model(model_name, model, X_clean, y_encoded, cv, selector):

    scaling_models = ["SVM", "Logistic Regression"]

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X_clean, y_encoded), 1):

        print("Fold:", fold)

        X_train = X_clean.iloc[train_idx]
        X_test = X_clean.iloc[test_idx]

        y_train = y_encoded[train_idx]
        y_test = y_encoded[test_idx]

        print("Train shape:", X_train.shape)
        print("Test shape:", X_test.shape)

        fold_selector = clone(selector)

        X_train_selected = fold_selector.fit_transform(X_train, y_train)
        X_test_selected = fold_selector.transform(X_test)

        selected_features = X_train.columns[fold_selector.get_support()]

        print("\nSelected Features:")
        print(selected_features.tolist())


        if model_name in scaling_models:
            scaler = StandardScaler()
            X_train_selected = scaler.fit_transform(X_train_selected)
            X_test_selected = scaler.transform(X_test_selected)

        fold_model = clone(model)

        fold_model.fit(X_train_selected, y_train)

        y_pred = fold_model.predict(X_test_selected)
        y_proba = fold_model.predict_proba(X_test_selected)[:, 1]

        fold_results.append({
            "Model": model_name,
            "Fold": fold,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1-score": f1_score(y_test, y_pred),
            "ROC-AUC": roc_auc_score(y_test, y_proba),
            # Weighted metrics
            "Weighted Precision": precision_score(y_test, y_pred, average="weighted"),
            "Weighted Recall": recall_score(y_test, y_pred, average="weighted"),
            "Weighted F1-score": f1_score(y_test, y_pred, average="weighted"),

            # Macro metrics
            "Macro Precision": precision_score(y_test, y_pred, average="macro"),
            "Macro Recall": recall_score(y_test, y_pred, average="macro"),
            "Macro F1-score": f1_score(y_test, y_pred, average="macro"),

            "ROC-AUC": roc_auc_score(y_test, y_proba)
})



    return pd.DataFrame(fold_results)

In [ ]:
rf_cv = run_cv_for_model("Random Forest", rf_model, X_clean, y_encoded, cv, selector)

xgb_cv = run_cv_for_model("XGBoost", xgb_model, X_clean, y_encoded, cv, selector)

lgbm_cv = run_cv_for_model("LightGBM", lgbm_model, X_clean, y_encoded, cv, selector)

cat_cv = run_cv_for_model("CatBoost", cat_model, X_clean, y_encoded, cv, selector)

svm_cv = run_cv_for_model("SVM", svm_model, X_clean, y_encoded, cv, selector)

lr_cv = run_cv_for_model("Logistic Regression", lr_model, X_clean, y_encoded, cv, selector)

Fold: 1
Train shape: (439, 604)
Test shape: (110, 604)

Selected Features:
['rs_CPB1', 'rs_SCGB2A2', 'rs_SCGB1D2', 'rs_TFF1', 'rs_PIP', 'rs_S100A7', 'rs_HMGCS2', 'rs_CYP2B7P1', 'rs_ANKRD30A', 'rs_PRAME', 'rs_SERPINA6', 'rs_AGR3', 'rs_DHRS2', 'rs_KCNJ3', 'rs_C4orf7', 'rs_GABRP', 'rs_SLC30A8', 'rs_STAC2', 'rs_VSTM2A', 'rs_CEACAM5', 'rs_CALML5', 'rs_GP2', 'rs_FABP7', 'rs_C1orf64', 'rs_CRABP1', 'rs_KRT6B', 'rs_CST9', 'rs_KLK5', 'rs_TFF3', 'rs_BMPR1B', 'rs_CEACAM6', 'rs_SLC34A2', 'rs_AGR2', 'rs_ABCC11', 'rs_GRIA2', 'rs_NPY1R', 'rs_C20orf114', 'rs_KIF1A', 'rs_TNNT1', 'rs_LRP2', 'rs_PROM1', 'rs_PVALB', 'rs_ELF5', 'rs_GFRA1', 'rs_CYP2A6', 'rs_PGR', 'rs_CGA', 'rs_VGLL1', 'rs_CP', 'rs_KCNC2', 'rs_MUC16', 'rs_KRT6A', 'rs_AQP5', 'rs_KLK6', 'rs_KLK7', 'rs_KRT81', 'rs_EEF1A2', 'rs_SERPINA11', 'rs_LBP', 'rs_MMP1', 'rs_BPIL1', 'rs_KRT16', 'rs_MSLN', 'rs_A2ML1', 'rs_SLC6A14', 'rs_PTPRT', 'rs_CA9', 'rs_MUC15', 'rs_SFRP1', 'rs_SLITRK6', 'rs_BEX1', 'rs_CYP4Z2P', 'rs_KLHDC7A', 'rs_SLC6A4', 'rs_C10orf82', '

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 331, number of negative: 108
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001797 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 40883
[LightGBM] [Info] Number of data points in the train set: 439, number of used features: 300
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Fold: 3
Train shape: (439, 604)
Test shape: (110, 604)

Selected Features:
['rs_CLEC3A', 'rs_CPB1', 'rs_SCGB2A2', 'rs_SCGB1D2', 'rs_TFF1', 'rs_PIP', 'rs_S100A7', 'rs_HMGCS2', 'rs_CYP2B7P1', 'rs_ANKRD30A', 'rs_P

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold: 5
Train shape: (440, 604)
Test shape: (109, 604)

Selected Features:
['rs_CLEC3A', 'rs_CPB1', 'rs_SCGB2A2', 'rs_SCGB1D2', 'rs_TFF1', 'rs_PIP', 'rs_S100A7', 'rs_HMGCS2', 'rs_CYP2B7P1', 'rs_ANKRD30A', 'rs_PRAME', 'rs_SERPINA6', 'rs_AGR3', 'rs_CYP4Z1', 'rs_DHRS2', 'rs_KCNJ3', 'rs_C4orf7', 'rs_GABRP', 'rs_SOX10', 'rs_SLC30A8', 'rs_STAC2', 'rs_VSTM2A', 'rs_CEACAM5', 'rs_CALML5', 'rs_GP2', 'rs_FABP7', 'rs_C1orf64', 'rs_CRABP1', 'rs_KRT6B', 'rs_CST9', 'rs_KLK5', 'rs_TFF3', 'rs_BMPR1B', 'rs_SLC34A2', 'rs_AGR2', 'rs_ABCC11', 'rs_GRIA2', 'rs_NPY1R', 'rs_C20orf114', 'rs_KIF1A', 'rs_LRP2', 'rs_PROM1', 'rs_PVALB', 'rs_ELF5', 'rs_GFRA1', 'rs_CYP2A6', 'rs_PGR', 'rs_CGA', 'rs_VGLL1', 'rs_PLIN1', 'rs_CP', 'rs_KCNC2', 'rs_MUC16', 'rs_KRT6A', 'rs_AQP5', 'rs_KLK6', 'rs_KLK7', 'rs_KRT81', 'rs_EEF1A2', 'rs_SERPINA11', 'rs_LBP', 'rs_MMP1', 'rs_BPIL1', 'rs_KRT16', 'rs_KLK10', 'rs_MSLN', 'rs_A2ML1', 'rs_SLC6A14', 'rs_PTPRT', 'rs_CA9', 'rs_MUC15', 'rs_SFRP1', 'rs_SLITRK6', 'rs_BEX1', 'rs_CYP4Z2P', 'rs_KLH

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold: 2
Train shape: (439, 604)
Test shape: (110, 604)

Selected Features:
['rs_CLEC3A', 'rs_CPB1', 'rs_SCGB2A2', 'rs_SCGB1D2', 'rs_TFF1', 'rs_PIP', 'rs_S100A7', 'rs_HMGCS2', 'rs_CYP2B7P1', 'rs_ANKRD30A', 'rs_PRAME', 'rs_SERPINA6', 'rs_AGR3', 'rs_CYP4Z1', 'rs_DHRS2', 'rs_KCNJ3', 'rs_C4orf7', 'rs_GABRP', 'rs_SOX10', 'rs_SLC30A8', 'rs_STAC2', 'rs_VSTM2A', 'rs_KRT5', 'rs_CEACAM5', 'rs_CALML5', 'rs_GP2', 'rs_FABP7', 'rs_C1orf64', 'rs_CRABP1', 'rs_KRT6B', 'rs_CST9', 'rs_KLK5', 'rs_TFF3', 'rs_BMPR1B', 'rs_CEACAM6', 'rs_SLC34A2', 'rs_AGR2', 'rs_ABCC11', 'rs_GRIA2', 'rs_KRT17', 'rs_NPY1R', 'rs_C20orf114', 'rs_KIF1A', 'rs_LRP2', 'rs_PROM1', 'rs_PVALB', 'rs_ELF5', 'rs_GFRA1', 'rs_CYP2A6', 'rs_PGR', 'rs_CGA', 'rs_VGLL1', 'rs_CP', 'rs_KCNC2', 'rs_MUC16', 'rs_KRT6A', 'rs_AQP5', 'rs_KLK6', 'rs_KLK7', 'rs_KRT81', 'rs_EEF1A2', 'rs_SERPINA11', 'rs_LBP', 'rs_MMP1', 'rs_BPIL1', 'rs_KRT16', 'rs_KLK10', 'rs_MSLN', 'rs_A2ML1', 'rs_SLC6A14', 'rs_PTPRT', 'rs_CA9', 'rs_MUC15', 'rs_SFRP1', 'rs_SLITRK6', 'rs_CYP

In [ ]:
print(rf_cv)

           Model  Fold  Accuracy  Balanced Accuracy  Precision    Recall  \
0  Random Forest     1  0.927273           0.889335   0.941176  0.963855   
1  Random Forest     2  0.927273           0.864346   0.921348  0.987952   
2  Random Forest     3  0.918182           0.883311   0.940476  0.951807   
3  Random Forest     4  0.936364           0.907854   0.952381  0.963855   
4  Random Forest     5  0.908257           0.852078   0.918605  0.963415   

   F1-score   ROC-AUC  Weighted Precision  Weighted Recall  Weighted F1-score  \
0  0.952381  0.956270            0.926160         0.927273           0.926307   
1  0.953488  0.942436            0.928965         0.927273           0.923996   
2  0.946108  0.966087            0.917324         0.918182           0.917655   
3  0.958084  0.959393            0.935748         0.936364           0.935954   
4  0.940476  0.905601            0.906457         0.908257           0.905679   

   Macro Precision  Macro Recall  Macro F1-score  
0    

In [ ]:
xgb_cv

,Model,Fold,Accuracy,Balanced Accuracy,Precision,Recall,F1-score,ROC-AUC,Weighted Precision,Weighted Recall,Weighted F1-score,Macro Precision,Macro Recall,Macro F1-score
0,XGBoost,1,0.927273,0.901830,0.951807,0.951807,0.951807,0.970995,0.927273,0.927273,0.927273,0.901830,0.901830,0.901830
1,XGBoost,2,0.927273,0.876841,0.931034,0.975904,0.952941,0.964748,0.926619,0.927273,0.925219,0.922039,0.876841,0.896471
2,XGBoost,3,0.918182,0.883311,0.940476,0.951807,0.946108,0.953592,0.917324,0.918182,0.917655,0.893315,0.883311,0.888148
3,XGBoost,4,0.927273,0.914324,0.962963,0.939759,0.951220,0.959393,0.929734,0.927273,0.928128,0.895275,0.914324,0.904181
4,XGBoost,5,0.899083,0.858401,0.927711,0.939024,0.933333,0.931798,0.897981,0.899083,0.898436,0.867702,0.858401,0.862893


In [ ]:
lgbm_cv

,Model,Fold,Accuracy,Balanced Accuracy,Precision,Recall,F1-score,ROC-AUC,Weighted Precision,Weighted Recall,Weighted F1-score,Macro Precision,Macro Recall,Macro F1-score
0,LightGBM,1,0.909091,0.889781,0.950617,0.927711,0.939024,0.966087,0.911955,0.909091,0.910160,0.871860,0.889781,0.880226
1,LightGBM,2,0.945455,0.913878,0.952941,0.975904,0.964286,0.953146,0.944856,0.945455,0.944730,0.936471,0.913878,0.924451
2,LightGBM,3,0.909091,0.877287,0.939759,0.939759,0.939759,0.968764,0.909091,0.909091,0.909091,0.877287,0.877287,0.877287
3,LightGBM,4,0.909091,0.889781,0.950617,0.927711,0.939024,0.962517,0.911955,0.909091,0.910160,0.871860,0.889781,0.880226
4,LightGBM,5,0.899083,0.858401,0.927711,0.939024,0.933333,0.920506,0.897981,0.899083,0.898436,0.867702,0.858401,0.862893


In [ ]:
cat_cv

,Model,Fold,Accuracy,Balanced Accuracy,Precision,Recall,F1-score,ROC-AUC,Weighted Precision,Weighted Recall,Weighted F1-score,Macro Precision,Macro Recall,Macro F1-score
0,CatBoost,1,0.918182,0.895805,0.951220,0.939759,0.945455,0.956716,0.919362,0.918182,0.918678,0.886324,0.895805,0.890909
1,CatBoost,2,0.936364,0.907854,0.952381,0.963855,0.958084,0.944668,0.935748,0.936364,0.935954,0.918498,0.907854,0.913004
2,CatBoost,3,0.918182,0.895805,0.951220,0.939759,0.945455,0.964748,0.919362,0.918182,0.918678,0.886324,0.895805,0.890909
3,CatBoost,4,0.890909,0.877733,0.949367,0.903614,0.925926,0.948684,0.898452,0.890909,0.893324,0.845651,0.877733,0.859515
4,CatBoost,5,0.880734,0.833785,0.915663,0.926829,0.921212,0.925023,0.879391,0.880734,0.879970,0.842447,0.833785,0.837965


In [ ]:
svm_cv

,Model,Fold,Accuracy,Balanced Accuracy,Precision,Recall,F1-score,ROC-AUC,Weighted Precision,Weighted Recall,Weighted F1-score,Macro Precision,Macro Recall,Macro F1-score
0,SVM,1,0.927273,0.901830,0.951807,0.951807,0.951807,0.955823,0.927273,0.927273,0.927273,0.901830,0.901830,0.901830
1,SVM,2,0.927273,0.876841,0.931034,0.975904,0.952941,0.926372,0.926619,0.927273,0.925219,0.922039,0.876841,0.896471
2,SVM,3,0.927273,0.901830,0.951807,0.951807,0.951807,0.956270,0.927273,0.927273,0.927273,0.901830,0.901830,0.901830
3,SVM,4,0.909091,0.889781,0.950617,0.927711,0.939024,0.955377,0.911955,0.909091,0.910160,0.871860,0.889781,0.880226
4,SVM,5,0.889908,0.839883,0.916667,0.939024,0.927711,0.885727,0.887768,0.889908,0.888454,0.858333,0.839883,0.848471


In [ ]:
lr_cv

,Model,Fold,Accuracy,Balanced Accuracy,Precision,Recall,F1-score,ROC-AUC,Weighted Precision,Weighted Recall,Weighted F1-score,Macro Precision,Macro Recall,Macro F1-score
0,Logistic Regression,1,0.909091,0.864793,0.929412,0.951807,0.940476,0.936635,0.907465,0.909091,0.907884,0.884706,0.864793,0.874084
1,Logistic Regression,2,0.881818,0.834226,0.916667,0.927711,0.922156,0.921464,0.880478,0.881818,0.881057,0.842949,0.834226,0.838436
2,Logistic Regression,3,0.872727,0.840696,0.925926,0.903614,0.914634,0.941098,0.876396,0.872727,0.874224,0.825032,0.840696,0.832317
3,Logistic Regression,4,0.863636,0.847166,0.935897,0.879518,0.906832,0.927265,0.874927,0.863636,0.867297,0.811699,0.847166,0.826298
4,Logistic Regression,5,0.871560,0.840108,0.925000,0.902439,0.913580,0.892502,0.875245,0.871560,0.873060,0.824569,0.840108,0.831790
